# Entrega 1 — Análise Comparativa de Representações

Este notebook compara BoW, TF-IDF e embeddings densos por similaridade de cosseno. O enunciado solicita os pares M003×M017, M008×M022 e M008×M031.

In [ ]:
from pathlib import Path
import json, pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ROOT=Path.cwd()
if not (ROOT/'data'/'manifestacoes.json').exists():
    ROOT=ROOT.parent
with open(ROOT/'data'/'manifestacoes.json',encoding='utf-8') as f:
    data=json.load(f)
df=pd.DataFrame(data)
df.head()


In [ ]:
textos=df.texto.tolist()
bow_vec=CountVectorizer(strip_accents="unicode")
X_bow=bow_vec.fit_transform(textos)
tfidf_vec=TfidfVectorizer(strip_accents="unicode",ngram_range=(1,2))
X_tfidf=tfidf_vec.fit_transform(textos)

pares=[("M003","M017"),("M008","M022"),("M008","M031")]
def score_sparse(X,a,b):
    ia=df.index[df.id==a][0]; ib=df.index[df.id==b][0]
    return float(cosine_similarity(X[ia],X[ib])[0,0])
result=[]
for a,b in pares:
    result.append({"par":f"{a} × {b}","BoW":score_sparse(X_bow,a,b),"TF-IDF":score_sparse(X_tfidf,a,b)})
pd.DataFrame(result)


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
model=SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
X_emb=model.encode(textos,normalize_embeddings=True,show_progress_bar=False)
result=[]
for a,b in pares:
    ia=df.index[df.id==a][0]; ib=df.index[df.id==b][0]
    result.append({"par":f"{a} × {b}","Embeddings":float(cosine_similarity([X_emb[ia]],[X_emb[ib]])[0,0])})
emb_df=pd.DataFrame(result)
emb_df


## Conclusão

BoW representa documentos pela presença/frequência de termos e tende a perder relações entre palavras com significados próximos quando o vocabulário utilizado muda. TF-IDF reduz o peso de termos muito frequentes e costuma melhorar a discriminação lexical, mas continua dependente da sobreposição de palavras. Embeddings densos procuram representar significado e contexto, sendo mais adequados para busca semântica e paráfrases. Os valores exatos devem ser obtidos executando as células com o dataset utilizado.